In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import SGD, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform
from skorch import NeuralNetRegressor
from transform import new_Xtrain, new_Xval, Y_train, Y_val, X_test, Y_test
from sklearn.decomposition import PCA

In [3]:
#Preparing the data
num_features = ['Exposure', 'VehPower', 'BonusMalus', 'VehAge_log', 'DrivAge_log', 'Density_log']
scaler = StandardScaler()
scaler.fit(new_Xtrain)
ready_Xtrain = scaler.transform(new_Xtrain)
ready_Xval = scaler.transform(new_Xval)
ready_test_X = scaler.transform(X_test)


In [4]:
class ModelDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [5]:
X_train = ready_Xtrain.astype("float32")
X_val = ready_Xval.astype("float32")
Y_train = Y_train.astype("float32")
Y_val = Y_val.astype("float32")

In [6]:
X_train_np = X_train#.to_numpy()
y_train_np = Y_train.to_numpy().reshape(-1, 1)

X_test_np = X_val#.to_numpy()
y_test_np = Y_val.to_numpy().reshape(-1, 1)

ds = ModelDataset(torch.from_numpy(X_train_np),torch.from_numpy(y_train_np))
ds_test = ModelDataset(torch.from_numpy(X_test_np),torch.from_numpy(y_test_np))

train_loader = DataLoader(ds, batch_size=516, shuffle=True)
test_loader = DataLoader(ds_test, batch_size=1, shuffle=True)

In [7]:
#This is our network

input_dim = X_train.shape[1]

class SimpleNN(nn.Module):
    def __init__(self, activation=nn.ReLU):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 36)
        self.fc2 = nn.Linear(36, 24)
        self.fc3 = nn.Linear(24,12)
        self.fc4 = nn.Linear(12, 1)
        self.activation = activation() 
    def forward(self, x):
        #x = x.view(-1, 64)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x
    
model = SimpleNN()

In [8]:
#We perform gridsearch to find the best parameters of the model
param_distributions = {
    'lr': [0.0001, 0.001],
    'optimizer': [Adam],
    'max_epochs': [10, 20, 30, 50],
    'batch_size': [128, 516],
    'module__activation': [nn.ReLU],
    'optimizer__weight_decay': [0.0, 0.0001 ],
}
from skorch import NeuralNetRegressor 
net = NeuralNetRegressor(module=SimpleNN)

grid_search = GridSearchCV(
    net,
    param_distributions,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train_np, y_train_np)

print(grid_search.best_params_)
best_net = grid_search.best_estimator_

Fitting 3 folds for each of 32 candidates, totalling 96 fits
  epoch    train_loss    valid_loss      dur
-------  ------------  ------------  -------
      1        0.0572        0.0530  13.3924
      2        0.0550        0.0522  12.5549
      3        0.0539        0.0509  13.3381
      4        0.0526        0.0497  12.6609
      5        0.0514        0.0486  12.4045
      6        0.0504        0.0479  10.8406
      7        0.0498        0.0475  11.5768
      8        0.0495        0.0474  13.2988
      9        0.0493        0.0473  10.6928
     10        0.0492        0.0472  12.3583
     11        0.0491        0.0472  10.7790
     12        0.0490        0.0471  10.7888
     13        0.0489        0.0471  11.0259
     14        0.0489        0.0470  10.3966
     15        0.0488        0.0470  11.3211
     16        0.0487        0.0470  11.2029
     17        0.0487        0.0470  10.7580
     18        0.0486        0.0469  10.7424
     19        0.0486        0.0469  10

In [ ]:
#with the best network found we do the prediction on the validation set
y_pred = best_net.predict(ready_Xval.astype("float32"))
print("Validation MSE:", mean_squared_error(Y_val, y_pred))
print("Validation RMSE:", np.sqrt(mean_squared_error(Y_val, y_pred)))
print("Validation MAE:", mean_absolute_error(Y_val, y_pred)) 
print("Validation R^2:", r2_score(Y_val, y_pred))

Validation MSE: 0.048274021595716476
Test RMSE: 0.21971349889280012
Test MAE: 0.08831293135881424
Validation R^2: 0.163554847240448


In [22]:
test_prediction = best_net.predict(ready_test_X.astype("float32"))
print("Test MSE:", mean_squared_error(Y_test, test_prediction))
print("Test RMSE:", np.sqrt(mean_squared_error(Y_test, test_prediction)))
print("Test MAE:", mean_absolute_error(Y_test, test_prediction)) 
print("Test R²:", r2_score(Y_test, test_prediction))

Test MSE: 0.0509212501347065
Test RMSE: 0.22565737332227037
Test MAE: 0.08792843669652939
Test R²: 0.15423625707626343
